# Amazon Product Reviews — Customer Behavior Analysis

In [8]:
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Load Data

In [9]:
df = pd.read_csv('../Datasets/Amazon_Product_Review.csv', encoding='latin1')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (1597, 27)


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,manufacturer,manufacturerNumber,name,prices,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,Amazon,NaN,Kindle Paperwhite,"[{""amountMax"":139.99,""amountMin"":139.99,""curre...",2015-08-08T00:00:00.000Z,NaN,139.0,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,Amazon,NaN,Kindle Paperwhite,"[{""amountMax"":139.99,""amountMin"":139.99,""curre...",2015-09-01T00:00:00.000Z,NaN,126.0,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,Amazon,NaN,Kindle Paperwhite,"[{""amountMax"":139.99,""amountMin"":139.99,""curre...",2015-07-20T00:00:00.000Z,NaN,69.0,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams


In [10]:
pd.DataFrame({
    'dtype'  : df.dtypes,
    'nulls'  : df.isnull().sum(),
    'null_%' : (df.isnull().mean() * 100).round(1),
    'unique' : df.nunique()
})

,dtype,nulls,null_%,unique
id,object,0,0.0,66
asins,object,0,0.0,54
brand,object,0,0.0,2
categories,object,0,0.0,19
colors,object,823,51.5,3
dateAdded,object,0,0.0,50
dateUpdated,object,0,0.0,42
dimension,object,1032,64.6,2
ean,float64,699,43.8,2
keys,object,0,0.0,66


## Data Cleaning

In [11]:
# Drop columns that are 100% null or irrelevant to behaviour analysis
df.drop(columns=[
    'reviews.userCity', 'reviews.userProvince',
    'sizes', 'ean', 'upc', 'manufacturerNumber', 'colors', 'dimension'
], inplace=True)
print('Remaining columns:', list(df.columns))

Remaining columns: ['id', 'asins', 'brand', 'categories', 'dateAdded', 'dateUpdated', 'keys', 'manufacturer', 'name', 'prices', 'reviews.date', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'weight']


In [12]:
# Parse dates
for col in ['reviews.date', 'dateAdded', 'dateUpdated']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

df['review_year']  = df['reviews.date'].dt.year
df['review_month'] = df['reviews.date'].dt.month
print('Review date range:', df['reviews.date'].min().date(), '—', df['reviews.date'].max().date())

Review date range: 2012-09-12 — 2017-07-31


In [13]:
# Extract USD price, sale flag, and currencies from the embedded JSON
def extract_usd_price(raw):
    try:
        entries = json.loads(raw)
        usd = [e for e in entries if e.get('currency') == 'USD']
        return (usd or entries)[0].get('amountMax', np.nan)
    except:
        return np.nan

def has_sale(raw):
    try:
        return any(str(e.get('isSale', 'false')).lower() == 'true' for e in json.loads(raw))
    except:
        return False

def extract_currencies(raw):
    try:
        return ', '.join(sorted({e.get('currency') for e in json.loads(raw) if e.get('currency')}))
    except:
        return np.nan

df['price_usd']  = df['prices'].apply(extract_usd_price)
df['on_sale']    = df['prices'].apply(has_sale)
df['currencies'] = df['prices'].apply(extract_currencies)
df['price_usd'].describe().round(2)

count    1597.00
mean       95.50
std        78.25
min         9.99
25%        39.99
50%        99.99
75%       129.99
max       464.00
Name: price_usd, dtype: float64

In [14]:
# Clean the noisy categories field — pull the first meaningful token
noise = {'mazon.co.uk', 'categories', 'see more', '', 'electronics features', 'consumer electronics'}

def clean_categories(raw):
    if not isinstance(raw, str):
        return 'Unknown'
    parts = [p.strip() for p in raw.split(',')]
    parts = [p for p in parts if p.lower() not in noise and not p.lower().startswith('see more')]
    return parts[0] if parts else 'Unknown'

df['primary_category'] = df['categories'].apply(clean_categories)
df['primary_category'].value_counts().head(10)

primary_category
Amazon Devices                  1365
Electronics                      111
Amazon Devices & Accessories      77
Kindle Store                      32
Cell Phones & Accessories         12
Name: count, dtype: int64

In [15]:
# Derive device type from product name
def get_device(name):
    n = str(name).lower()
    if 'echo' in n:                       return 'Echo'
    elif 'fire tv' in n:                  return 'Fire TV'
    elif 'fire hd' in n or 'hdx' in n:   return 'Fire Tablet'
    elif 'kindle' in n:                   return 'Kindle'
    elif 'tap' in n:                      return 'Amazon Tap'
    elif 'fire' in n:                     return 'Fire'
    return 'Accessory'

df['device_type'] = df['name'].apply(get_device)
df['device_type'].value_counts()

device_type
Amazon Tap     557
Fire Tablet    309
Fire TV        251
Kindle         182
Accessory      133
Echo            98
Fire            67
Name: count, dtype: int64

In [16]:
# Clean review text — strip HTML, non-ASCII, and extra whitespace
def clean_text(txt):
    if not isinstance(txt, str): return ''
    txt = re.sub(r'<[^>]+>', ' ', txt)
    txt = re.sub(r'[^\x00-\x7F]+', ' ', txt)
    return re.sub(r'\s+', ' ', txt).strip()

df['reviews.text']  = df['reviews.text'].apply(clean_text)
df['reviews.title'] = df['reviews.title'].apply(clean_text)

# Fill missing ratings with per-product median, then overall median
df['reviews.rating'] = df.groupby('name')['reviews.rating'].transform(lambda x: x.fillna(x.median()))
df['reviews.rating'].fillna(df['reviews.rating'].median(), inplace=True)

df['reviews.username'].fillna('Anonymous', inplace=True)
df['reviews.title'].fillna('No Title', inplace=True)
df['reviews.doRecommend'] = df['reviews.doRecommend'].map({True: True, False: False, 'TRUE': True, 'FALSE': False})

print('Nulls remaining:')
remaining = df.isnull().sum()
print(remaining[remaining > 0])

Nulls remaining:
manufacturer            632
reviews.date            747
reviews.doRecommend    1058
reviews.numHelpful      697
weight                  911
review_year             747
review_month            747
dtype: int64


In [17]:
# Sentiment labels
df['sentiment'] = df['reviews.rating'].apply(
    lambda r: 'Positive' if r >= 4 else ('Neutral' if r == 3 else 'Negative')
)

pos_kw = ['love', 'great', 'excellent', 'amazing', 'perfect', 'awesome', 'best']
neg_kw = ['terrible', 'awful', 'worst', 'horrible', 'broken', 'defective', 'disappointed']

def keyword_sentiment(text):
    t = text.lower()
    pos = any(w in t for w in pos_kw)
    neg = any(w in t for w in neg_kw)
    if pos and not neg:  return 'Positive'
    if neg and not pos:  return 'Negative'
    return 'Neutral'

df['text_sentiment'] = df['reviews.text'].apply(keyword_sentiment)
print(df['sentiment'].value_counts())
print(df['text_sentiment'].value_counts())

sentiment
Positive    1369
Neutral      152
Negative      76
Name: count, dtype: int64
text_sentiment
Positive    1029
Neutral      550
Negative      18
Name: count, dtype: int64


## Save Cleaned Dataset

In [ ]:
df.to_csv('../Datasets/Amazon_Product_Review_Cleaned.csv', index=False)

print('Saved: Datasets/Amazon_Product_Review_Cleaned.csv')
print(f'Shape: {df.shape}')
print(f'\nColumns in cleaned file:')
print(list(df.columns))